In [10]:
import math
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout

In [11]:
df = yf.download('AAPL', start='2015-01-01', end='2024-01-01')

/tmp/ipykernel_19842/1666373606.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('AAPL', start='2015-01-01', end='2024-01-01')
[*********************100%***********************]  1 of 1 completed


In [12]:
dataset = df['Close'].values
dataset = dataset.reshape(-1, 1)

In [13]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(dataset)

In [14]:
training_data_len = math.ceil(len(dataset) * .8)
lookback = 60

train_data = scaled_data[0:training_data_len, :]
x_train, y_train = [], []

for i in range(lookback, len(train_data)):
    x_train.append(train_data[i-lookback:i, 0])
    y_train.append(train_data[i, 0])

x_train, y_train = np.array(x_train), np.array(y_train)

In [15]:
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

In [16]:
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(Dropout(0.2))
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=25))
model.add(Dense(units=1))
model.compile(optimizer='adam', loss='mean_squared_error')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [17]:
print("Training model:")
model.fit(x_train, y_train, batch_size=32, epochs=10)

Training model:
Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - loss: 0.0085
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - loss: 0.0016
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - loss: 0.0013
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - loss: 0.0012
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - loss: 9.0387e-04
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 9.1892e-04
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 74ms/step - loss: 8.3877e-04
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - loss: 0.0010
Epoch 9/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 8.3728e-04
Epoch 10/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - loss: 8.7916e-04
